# Pipelines and Joins

This notebook covers the two composition operators and the `JoinModule` for multi-frame pipelines:

- `|` (pipe) — creates a `SequentialModule` where each step receives the previous step's output as `"input"`
- `&` (merge) — creates a `UnionExpressionModule` that compiles all expression nodes in a single Polars pass
- `JoinModule` — joins two named input frames before passing the result downstream

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '../../')

import polars as pl
from decider.modules.functional import generate_from_functions
from decider.modules.primitives.join import JoinModule
from decider.modules.primitives.sequential import SequentialModule

print('Imports OK')

## Sequential Pipelines with `|`

Use `|` to chain modules. The output of each step becomes the `"input"` frame for the next step, so downstream modules can reference columns produced upstream.

In [ ]:
# Step 1 — feature engineering
def log_amount(amount: pl.Expr) -> pl.Expr:
    return amount.log(base=10)

def amount_squared(amount: pl.Expr) -> pl.Expr:
    return amount ** 2

Features = generate_from_functions('features', log_amount, amount_squared)
features = Features(name='features')

# Step 2 — scoring (reads log_amount produced by step 1)
def raw_score(log_amount: pl.Expr) -> pl.Expr:
    return log_amount * pl.lit(50.0)

Scoring = generate_from_functions('scoring', raw_score)
scoring = Scoring(name='scoring')

# Step 3 — flags (reads raw_score produced by step 2)
def high_value_flag(raw_score: pl.Expr) -> pl.Expr:
    return raw_score > pl.lit(80.0)

def score_band(raw_score: pl.Expr) -> pl.Expr:
    return (
        pl.when(raw_score >= pl.lit(100.0)).then(pl.lit('platinum'))
        .when(raw_score >= pl.lit(80.0)).then(pl.lit('gold'))
        .otherwise(pl.lit('standard'))
    )

Flags = generate_from_functions('flags', high_value_flag, score_band)
flags = Flags(name='flags')

# Chain all three steps
pipeline = features | scoring | flags

print('Pipeline type :', type(pipeline).__name__)
print('Steps         :', [s.name for s in pipeline.steps])

In [ ]:
df = pl.DataFrame({
    'customer_id': ['C1', 'C2', 'C3', 'C4'],
    'amount':      [10.0, 100.0, 1000.0, 10000.0],
})

result = pipeline({'input': df})
print(result)

Each step's output columns are forwarded to the next step. `scoring` can read `log_amount` because step 1 produced it; `flags` can read `raw_score` because step 2 produced it.

## Merging Parallel Branches with `&`

Use `&` to merge two independent `ExpressionModule` branches. All expression nodes from both modules are compiled into a single Polars `.with_columns()` pass — more efficient than two separate steps.

In [ ]:
# Branch A — transaction features
def txn_velocity(txn_count: pl.Expr) -> pl.Expr:
    """How fast is this customer transacting?"""
    return txn_count / pl.lit(30.0)  # transactions per day over 30-day window

def avg_txn_value(total_spend: pl.Expr, txn_count: pl.Expr) -> pl.Expr:
    return total_spend / txn_count

TxnFeatures = generate_from_functions('txn_features', txn_velocity, avg_txn_value)
txn_features = TxnFeatures(name='txn_features')

# Branch B — behavioural features (independent of Branch A)
def login_rate(login_count: pl.Expr) -> pl.Expr:
    return login_count / pl.lit(30.0)

def engagement_score(login_count: pl.Expr, txn_count: pl.Expr) -> pl.Expr:
    return (login_count * pl.lit(0.4)) + (txn_count * pl.lit(0.6))

BehavFeatures = generate_from_functions('behav_features', login_rate, engagement_score)
behav_features = BehavFeatures(name='behav_features')

# Merge both branches — single Polars pass
merged = txn_features & behav_features

print('Merged type:', type(merged).__name__)

customer_df = pl.DataFrame({
    'customer_id':  ['C1', 'C2', 'C3'],
    'txn_count':    [5,    20,   3],
    'total_spend':  [250.0, 1800.0, 90.0],
    'login_count':  [12,   30,    2],
})

result_merged = merged({'input': customer_df})
print(result_merged)

## JoinModule

`JoinModule` joins two named input frames before passing the combined frame downstream. The `left` and `right` fields are string keys into the input dict.

In [ ]:
# Two separate frames
txns_df = pl.DataFrame({
    'user_id':   ['U1', 'U1', 'U2', 'U3'],
    'txn_id':    ['T1', 'T2', 'T3', 'T4'],
    'amount':    [50.0, 30.0, 200.0, 15.0],
})

users_df = pl.DataFrame({
    'user_id':    ['U1',    'U2',      'U3'],
    'credit_tier': ['gold', 'platinum', 'standard'],
    'income':      [60000.0, 120000.0, 35000.0],
})

# Join step
join = JoinModule(
    name='join_txns_users',
    left='txns',
    right='users',
    on='user_id',
    how='left',
)

# Scorer runs on the joined frame
def spend_to_income_ratio(amount: pl.Expr, income: pl.Expr) -> pl.Expr:
    return amount / income

TxnScorer = generate_from_functions('txn_scorer', spend_to_income_ratio)
txn_scorer = TxnScorer(name='txn_scorer')

# Chain: join first, then score
join_pipeline = join | txn_scorer

result_join = join_pipeline({'txns': txns_df, 'users': users_df})
print(result_join)

## Multi-Frame Input: `get_input_frame_keys()`

You can inspect which frame keys a pipeline expects from its input dict. This is useful for documentation and validation.

In [ ]:
print('Single scorer input keys :', txn_scorer.get_input_frame_keys())
print('Join module input keys   :', join.get_input_frame_keys())
print('Full pipeline input keys :', join_pipeline.get_input_frame_keys())